# Aumento de datos

## Importaciones y constantes

Se importaron las bibliotecas necesarias para el funcionamiento del script. `random` proporcionó los mecanismos de aleatoriedad, `argparse` gestionó los argumentos de la terminal, `pathlib` manejó las rutas del sistema de archivos, `numpy` permitió la manipulación matricial de los píxeles, y `PIL` proveyó las herramientas de procesamiento de imágenes. Adicionalmente, se definieron las extensiones de archivo reconocidas como imágenes válidas y el número predeterminado de versiones a generar.

In [ ]:
import random
import argparse
from pathlib import Path

import numpy as np
from PIL import Image, ImageEnhance, ImageFilter

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
NUM_AUGMENTATIONS = 10

: 

## Transformaciones individuales

Se implementaron siete funciones de transformación, cada una encargada de modificar un aspecto específico de la imagen. Las alteraciones aplicadas comprendieron rotación, volteo horizontal, ajuste de brillo, ajuste de contraste, adición de ruido, desenfoque gaussiano y recorte aleatorio con redimensionamiento. Cada función recibió una imagen y devolvió una versión modificada de la misma.

In [ ]:
def random_rotation(img):
    angle = random.uniform(-30, 30)
    return img.rotate(angle, expand=True, fillcolor=(128, 128, 128))

def random_flip(img):
    return img.transpose(Image.FLIP_LEFT_RIGHT) if random.random() > 0.5 else img

def random_brightness(img):
    return ImageEnhance.Brightness(img).enhance(random.uniform(0.5, 1.5))

def random_contrast(img):
    return ImageEnhance.Contrast(img).enhance(random.uniform(0.5, 1.5))

def random_noise(img):
    arr = np.array(img).astype(np.int16)
    noise = np.random.randint(-25, 25, arr.shape, dtype=np.int16)
    return Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))

def random_blur(img):
    return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 2.0)))

def random_crop(img, min_ratio=0.7):
    w, h = img.size
    crop_w = int(random.uniform(min_ratio, 1.0) * w)
    crop_h = int(random.uniform(min_ratio, 1.0) * h)
    x = random.randint(0, w - crop_w)
    y = random.randint(0, h - crop_h)
    return img.crop((x, y, x + crop_w, y + crop_h)).resize((w, h), Image.LANCZOS)


TRANSFORMS = [random_rotation, random_flip, random_brightness,
              random_contrast, random_noise, random_blur, random_crop]

## Función de aumento

Se definió una función que seleccionó aleatoriamente entre 2 y 7 de las transformaciones disponibles y las aplicó de forma secuencial sobre la imagen. Este mecanismo garantizó que cada versión generada fuera única e irrepetible.

In [ ]:
def augment(img):
    for transform in random.sample(TRANSFORMS, k=random.randint(2, len(TRANSFORMS))):
        img = transform(img)
    return img

## Función de procesamiento por imagen

Se implementó la lógica encargada de procesar una imagen individual. Por cada imagen se creó una subcarpeta con su mismo nombre, dentro de la cual se almacenó el archivo original y cada una de las versiones aumentadas, nombradas secuencialmente como `aug_1`, `aug_2`, y así sucesivamente.

In [ ]:
def process_image(image_path, num_augmentations):
    img = Image.open(image_path).convert("RGB")
    out_dir = image_path.parent / image_path.stem
    out_dir.mkdir(parents=True, exist_ok=True)

    img.save(out_dir / f"original{image_path.suffix}")
    print(f"  original -> {out_dir / f'original{image_path.suffix}'}")

    for i in range(1, num_augmentations + 1):
        out_path = out_dir / f"aug_{i}{image_path.suffix}"
        augment(img.copy()).save(out_path)
        print(f"  aug_{i} -> {out_path}")

## Función principal

Se configuró la interfaz de línea de comandos mediante `argparse`, la cual aceptó la ruta de la carpeta de entrada y el parámetro opcional `--cantidad`. Posteriormente, se validó la existencia de la carpeta, se recopilaron las imágenes válidas y se invocó el procesamiento de cada una, registrando en consola el progreso y los errores encontrados.

In [ ]:
def main():
    parser = argparse.ArgumentParser(
        description="Genera versiones aumentadas de cada imagen en la carpeta indicada."
    )
    parser.add_argument("carpeta", help="Ruta a la carpeta con imagenes")
    parser.add_argument("--cantidad", type=int, default=10, metavar="N",
                        help="Versiones aumentadas por imagen (por defecto: 10)")

    args = parser.parse_args()
    input_path = Path(args.carpeta)

    if not input_path.is_dir():
        print(f"Error: '{args.carpeta}' no es una carpeta valida.")
        return

    images = sorted(p for p in input_path.iterdir()
                    if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS)

    if not images:
        print(f"No se encontraron imagenes en '{args.carpeta}'.")
        return

    print(f"Imagenes encontradas: {len(images)}\n")

    for image_path in images:
        print(f"Procesando: {image_path.name}")
        try:
            process_image(image_path, args.cantidad)
        except Exception as e:
            print(f"  Error al procesar {image_path.name}: {e}")
        print()

    print("Proceso completado.")


if __name__ == "__main__":
    main()